In [ ]:
!pip install ragas

In [ ]:
!pip install langchain_huggingface

In [1]:
import os
import getpass
def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

In [2]:
_set_env("OPENAI_API_KEY")

OPENAI_API_KEY:  ········


## Проверка датасета

In [1]:
import pandas as pd
import json
import ast
import chromadb
import time
import numpy as np
import math
from langchain_chroma import Chroma

from chromadb.utils import embedding_functions
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

from sentence_transformers import SentenceTransformer
from tqdm import tqdm

In [2]:
import DatasetEvaluation

In [ ]:
emb_name = "intfloat/multilingual-e5-large"
dataset_name = "2025_11_10 QAnswers.csv"

In [4]:
semantic_search_test = DatasetEvaluation.DatasetEvaluation()

In [5]:
start_time = time.time()

semantic_search_test.evaluate_dataset(emb_name,dataset_name,58)

response_time = time.time() - start_time
print(f"Время создания: {response_time:.3f} секунды")

100%|█████████████████████████████████████████████████████████████████████████████████| 199/199 [02:00<00:00,  1.66it/s]

Вопросов всего: 1197
Правильных индексов (первый верный): 540 доля: 45.11%
Первый или второй верные: 686 доля: 57.31%
Индекс есть в списке из 5: 847 доля: 70.76%
Время создания: 124.335 секунды


In [5]:
table = pd.read_csv(dataset_name, sep=";")

In [12]:
table.iloc[58]

Unnamed: 0                                                   58
Number                                                      3.2
Header        Контрольные сроки выполнения ключевых работ в ...
Text          Основные сроки и виды работ представлены в таб...
Clean_Text    Основные сроки и виды работ представлены в таб...
HTML          ```html\n<!DOCTYPE html>\n<html lang="ru">\n<h...
File_Name                                              058.html
Json_Q_A      \n[\n    {\n        "Question": "Что такое ЭТЖ...
Answers       ['ЭТЖ — это аббревиатура, которая расшифровыва...
Indexes       ['199 199 158 1 161', '199 215 58 199 158', '1...
Name: 58, dtype: object

In [50]:
table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 257 entries, 0 to 256
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Unnamed: 0        257 non-null    int64 
 1   Number            257 non-null    object
 2   Header            257 non-null    object
 3   Text              257 non-null    object
 4   Clean_Text        257 non-null    object
 5   HTML              257 non-null    object
 6   File_Name         257 non-null    object
 7   Json_Q_A          257 non-null    object
 8   HeadersHierarchy  257 non-null    object
dtypes: int64(1), object(8)
memory usage: 18.2+ KB


In [44]:
dataset=semantic_search_test.get_dataset(table,57)

100%|████████████████████████████████████████████████████████████████████████████████| 200/200 [00:00<00:00, 516.36it/s]
1197it [00:00, 855878.26it/s]


### полный прогон

In [ ]:
result=semantic_search_test.evaluate_metrics(dataset)

In [32]:
result.total_tokens()

TokenUsage(input_tokens=14523812, output_tokens=1694507, model='')

In [33]:
result.total_cost(cost_per_input_token=0.02 /1000, cost_per_output_token=0.08 / 1000)

426.0368

In [29]:
result

{'llm_context_precision_with_reference': 0.7580, 'context_recall': 0.8011, 'faithfulness': 0.4902, 'answer_relevancy': 0.6495, 'context_entity_recall': 0.1787}

## сохранение средних значений таблицу

In [30]:
semantic_search_test.save_results(False)

### Сохранение полных результатов метрик

In [ ]:
semantic_search_test.save_results(True)

Context Precision (точность) - сколько правильных найдено (нужно чтобы находил несколько)
Context Recall (полнота) - была ли извлечена вся необходимая для ответа на вопрос информация. It focuses on not missing important results.
Faithfulness (верность) - точность сгенерированного ответа (правильные утверждения в ответе/все утверждения в ответе)
Response Relevancy (Релевантность ответа) - насколько релевантен сгенерированный ответ вопросу пользователя.
Semantic similarity (семантическое сходство ответов) - оценивает семантическое сходство между сгенерированным ответом и истинным значением.based on the ground truth and the answer
FactualCorrectness (Корректность ответа) - измерение точности сгенерированного ответа по сравнению с истинным значением. This metric is used to determine the extent to which the generated response aligns with the reference.
Context Entities Recall (Возврат объекта контекста) - предоставляет меру возврата извлеченного контекста на основе количества объектов, присутствующих как в истине, так и в контекстах, относительно количества объектов, присутствующих только в истине. (например количество терминов чтобы совпало)
Noise Sensitivity -  measures how often a system makes errors by providing incorrect responses when utilizing either relevant or irrelevant retrieved documents.
Summarization Score - gives a measure of how well the summary (response) captures the important information from the retrieved_contexts. The intuition behind this metric is that a good summary shall contain all the important information present in the context(or text so to say).
Aspect Critic - банальный промпт для ллм настроенный на конкретную тему (проверка токсичности например)
есть метрики для ранкинга(но не в рагасе)
NDCG, MAP